# 批量提取洪水事件（daily ts 批处理 Notebook）

这个 Notebook 将**递归遍历**你提供的根目录，自动寻找名字里包含 `daily` 和 `ts` 的 CSV（例如 `*_daily_ts.csv`），
并基于你提供的逻辑从径流时间序列中提取**每年**的**洪水事件完整时间序列**（起涨—峰值—衰退），输出：

- 每站点 `flood_events_summary.csv`（事件摘要）
- 每站点 `flood_events_timeseries.csv`（事件内逐日曲线，含 `flag` 标注）
- 全站汇总 `ALL_flood_events_summary.csv`、`ALL_flood_events_timeseries.csv`
- `batch_log.csv` 记录每个文件处理状态

你可以在下方 **【参数设置】** 中指定根目录与输出目录，然后 **运行全部单元格**。


In [1]:

# === 导入依赖 ===
import os, re
from pathlib import Path
import pandas as pd
import numpy as np
from concurrent.futures import ProcessPoolExecutor, as_completed

# === 基础工具 ===
def derive_station_name_from_filename(path: Path) -> str:
    """
    从文件名推断站点名，去掉前缀站码与后缀。
    例如：003303A_Barcoo_River_at_Blackall_003303A_daily_ts.csv
          -> "Barcoo River at Blackall 003303A"
    """
    stem = path.stem  # 去掉 .csv
    parts = stem.split("_")
    # 抓取 "daily" 出现前的 token
    if "daily" in parts:
        idx = parts.index("daily")
        core = parts[:idx]
    else:
        core = parts

    # 去掉第一个类似站码的 token（如 003303A）
    if len(core) > 1 and re.match(r"^[A-Za-z]*\d+[A-Za-z]*$", core[0]):
        core = core[1:]

    name = " ".join(core).strip()
    return name if name else stem

def read_daily_ts_csv(path: Path) -> pd.DataFrame:
    """
    读取类似 BOM/HRS 的 daily ts CSV，兼容注释头（#），返回标准列：['time','runoff','station_name']
    """
    df0 = pd.read_csv(path, comment="#", engine="python", on_bad_lines="skip")
    # 常见列名映射候选
    colmap_candidates = [
        ("Date", "Flow (ML)"),
        ("Date", "Flow (ML/day)"),
        ("date", "flow"),
        ("DATE", "FLOW"),
    ]
    time_col, flow_col = None, None
    cols_lower = {c.lower(): c for c in df0.columns}
    for tc, fc in colmap_candidates:
        if tc in df0.columns and fc in df0.columns:
            time_col, flow_col = tc, fc
            break
    if time_col is None:
        # 小写兜底
        for tc, fc in [("date","flow"), ("date","flow (ml)"), ("date","flow (ml/day)")]:  # noqa
            if tc in cols_lower and fc in cols_lower:
                time_col, flow_col = cols_lower[tc], cols_lower[fc]
                break
    if time_col is None or flow_col is None:
        # 最后兜底：前两列
        if len(df0.columns) >= 2:
            time_col, flow_col = df0.columns[0], df0.columns[1]
        else:
            raise ValueError(f"Cannot find time/flow columns in {path}")

    df = pd.DataFrame({
        "time": pd.to_datetime(df0[time_col], errors="coerce"),
        "runoff": pd.to_numeric(df0[flow_col], errors="coerce"),
    }).dropna(subset=["time"])

    df["station_name"] = derive_station_name_from_filename(path)
    return df

def looks_like_daily_ts(filename: str) -> bool:
    """
    宽松匹配：文件名含 'daily' 与 'ts'（顺序不限），且以 .csv 结尾。
    可根据你的命名习惯改成更严格的规则。
    """
    if not filename.lower().endswith(".csv"):
        return False
    s = filename.lower()
    return ("daily" in s) and ("ts" in s)

# === 事件提取核心（与你的逻辑一致，增加健壮性） ===
def extract_flood_events_timeseries(df, tol=0.05, min_valid_ratio=0.5):
    """
    从径流时间序列中提取洪水事件完整时间序列
    期望 df 列：['time','runoff','station_name']
    """
    x = df.dropna(subset=["runoff"]).copy()
    if x.empty:
        return []
    x["year"] = x["time"].dt.year

    events = []
    event_id_counter = 1

    for year, group in x.groupby("year", sort=True):
        group = group.sort_values("time").reset_index(drop=True)

        # 缺测比例判断（以该年的总长度计）
        n_total = len(group)
        n_valid = group["runoff"].notna().sum()
        if n_total == 0 or n_valid / n_total < min_valid_ratio:
            continue

        # 峰值
        idx_peak = int(group["runoff"].idxmax())
        peak_time = group.loc[idx_peak, "time"]
        peak_value = float(group.loc[idx_peak, "runoff"])

        # 起涨点：向左回溯，直到不再非降（前值 <= 当前值）
        idx_rising = idx_peak
        while idx_rising > 0 and group.loc[idx_rising - 1, "runoff"] <= group.loc[idx_rising, "runoff"]:
            idx_rising -= 1
        rising_time = group.loc[idx_rising, "time"]
        rising_value = float(group.loc[idx_rising, "runoff"])

        # 衰退点：向右到起涨点±tol（阈值 = rising_value * (1+tol)）
        idx_recession = idx_peak
        threshold = rising_value * (1 + tol)
        while idx_recession < len(group) - 1 and group.loc[idx_recession, "runoff"] > threshold:
            idx_recession += 1
        if idx_recession >= len(group):
            idx_recession = len(group) - 1

        recession_time = group.loc[idx_recession, "time"]
        recession_value = float(group.loc[idx_recession, "runoff"])

        # 打包过程
        flood_process = group.loc[idx_rising:idx_recession, ["time","runoff"]].copy()
        station_name = group["station_name"].iloc[0] if "station_name" in group.columns else ""
        eid_label = f"{station_name}_{year}_{event_id_counter}" if station_name else f"{year}_{event_id_counter}"
        flood_process["station_name"] = station_name
        flood_process["event_id"] = eid_label
        flood_process["flag"] = "process"
        flood_process.loc[flood_process.index[0], "flag"] = "rising"
        flood_process.loc[flood_process.index[-1], "flag"] = "recession"
        # 峰值标记
        if idx_rising <= idx_peak <= idx_recession:
            flood_process.loc[flood_process.index[idx_peak - idx_rising], "flag"] = "peak"

        events.append({
            "year": int(year),
            "event_id": event_id_counter,
            "event_id_label": eid_label,
            "rising_time": rising_time,
            "rising_value": rising_value,
            "peak_time": peak_time,
            "peak_value": peak_value,
            "recession_time": recession_time,
            "recession_value": recession_value,
            "data": flood_process
        })
        event_id_counter += 1

    return events

# === 单文件处理 ===
def process_one_file(path: Path, out_dir: Path, tol=0.05, min_valid_ratio=0.5):
    """处理单个 daily ts 文件，返回 (summary_df, timeseries_df, meta_log)"""
    try:
        df = read_daily_ts_csv(path)
        events = extract_flood_events_timeseries(df, tol=tol, min_valid_ratio=min_valid_ratio)
        if not events:
            return None, None, {"file": str(path), "status": "no_events"}

        all_ts = pd.concat([e["data"] for e in events], ignore_index=True)
        summary = pd.DataFrame([{
            "source_file": str(path),
            "station_name": e["data"]["station_name"].iloc[0],
            "year": e["year"],
            "event_id": e["event_id_label"],
            "rising_time": e["rising_time"],
            "rising_value(ML/day)": e["rising_value"],
            "peak_time": e["peak_time"],
            "peak_value(ML/day)": e["peak_value"],
            "recession_time": e["recession_time"],
            "recession_value(ML/day)": e["recession_value"],
            "n_points": len(e["data"]),
            "duration_days": (e["recession_time"] - e["rising_time"]).days + 1
        } for e in events])

        # 写出到 per-station 文件夹
        st = df["station_name"].iloc[0] if "station_name" in df.columns else "UnknownStation"
        st_safe = re.sub(r"[^A-Za-z0-9_\-]+", "_", st)
        st_dir = out_dir / st_safe
        st_dir.mkdir(parents=True, exist_ok=True)

        summary_path = st_dir / f"{st_safe}_flood_events_summary.csv"
        ts_path = st_dir / f"{st_safe}_flood_events_timeseries.csv"
        summary.to_csv(summary_path, index=False)
        all_ts.to_csv(ts_path, index=False)

        return summary, all_ts, {"file": str(path), "status": "ok", "n_events": len(summary)}
    except Exception as e:
        return None, None, {"file": str(path), "status": "error", "error": str(e)}

# === 批处理主函数 ===
def run_batch(root: str, out_dir: str, tol=0.05, min_valid_ratio=0.5, workers=4):
    root_p = Path(root)
    out_p = Path(out_dir)
    out_p.mkdir(parents=True, exist_ok=True)

    # 递归搜集候选文件
    candidates = []
    for p in root_p.rglob("*.csv"):
        if looks_like_daily_ts(p.name):
            candidates.append(p)

    logs = []
    all_summaries, all_timeseries = [], []

    if workers and workers > 1:
        with ProcessPoolExecutor(max_workers=workers) as ex:
            fut2path = {ex.submit(process_one_file, p, out_p, tol, min_valid_ratio): p for p in candidates}
            for fut in as_completed(fut2path):
                summary, ts, meta = fut.result()
                logs.append(meta)
                if summary is not None:
                    all_summaries.append(summary)
                if ts is not None:
                    all_timeseries.append(ts)
    else:
        for p in candidates:
            summary, ts, meta = process_one_file(p, out_p, tol=tol, min_valid_ratio=min_valid_ratio)
            logs.append(meta)
            if summary is not None:
                all_summaries.append(summary)
            if ts is not None:
                all_timeseries.append(ts)

    # 全局汇总
    global_summary = pd.concat(all_summaries, ignore_index=True) if all_summaries else pd.DataFrame()
    global_ts = pd.concat(all_timeseries, ignore_index=True) if all_timeseries else pd.DataFrame()
    log_df = pd.DataFrame(logs)

    # 写出全局文件
    if not global_summary.empty:
        global_summary.to_csv(out_p / "ALL_flood_events_summary.csv", index=False)
    if not global_ts.empty:
        global_ts.to_csv(out_p / "ALL_flood_events_timeseries.csv", index=False)
    log_df.to_csv(out_p / "batch_log.csv", index=False)

    return {
        "n_files_found": len(candidates),
        "n_ok": int((log_df["status"] == "ok").sum() if "status" in log_df.columns else 0),
        "n_errors": int((log_df["status"] == "error").sum() if "status" in log_df.columns else 0),
        "out_dir": str(out_p),
    }


## 参数设置
把下面的 `ROOT_DIR` 改成你的**数据根目录**（会递归遍历），`OUT_DIR` 改成**输出目录**。
其他参数：
- `TOL`：衰退判定阈值（阈值 = 起涨值 × (1 + TOL)）
- `MIN_VALID_RATIO`：每年最小有效比例（例如 0.5 表示有效点数/全年点数 ≥ 0.5 才处理）
- `WORKERS`：并行进程数（≥1；没有并行请设 1）


In [11]:
# === 修改这里 ===
ROOT_DIR = r"D:\workroom\GPLW\makeUP\downloaded_stations_data"
OUT_DIR  = r"D:\workroom\GPLW\makeUP\flood_out"

TOL = 0.1
MIN_VALID_RATIO = 0.5
WORKERS = 1 # 可根据机器改为 6/8 等


## 执行批处理
运行下方单元格开始处理（会输出统计信息并在输出目录生成多个 CSV 文件）。


In [13]:

stats = run_batch(
    root=ROOT_DIR,
    out_dir=OUT_DIR,
    tol=TOL,
    min_valid_ratio=MIN_VALID_RATIO,
    workers=WORKERS
)
stats


C:\Users\huawei\AppData\Local\Temp\ipykernel_14628\1616639921.py:63: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  "time": pd.to_datetime(df0[time_col], errors="coerce"),
C:\Users\huawei\AppData\Local\Temp\ipykernel_14628\1616639921.py:63: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  "time": pd.to_datetime(df0[time_col], errors="coerce"),
C:\Users\huawei\AppData\Local\Temp\ipykernel_14628\1616639921.py:63: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  "time": pd.to_datetime(df0[time_col], errors="coerce"),
C:\Users\huawei\AppData\Local\Temp\ipykernel_14628\1616639921.py:63

{'n_files_found': 495,
 'n_ok': 487,
 'n_errors': 0,
 'out_dir': 'D:\\workroom\\GPLW\\makeUP\\flood_out'}

## （可选）快速查看输出
如果你已经运行完上面的单元格，可以用下面的代码预览全局汇总的前几行（若文件存在）。


In [14]:
import pandas as pd, os
sum_path = os.path.join(OUT_DIR, "ALL_flood_events_summary.csv")
ts_path  = os.path.join(OUT_DIR, "ALL_flood_events_timeseries.csv")
log_path = os.path.join(OUT_DIR, "batch_log.csv")

if os.path.exists(sum_path):
    display(pd.read_csv(sum_path).head())
else:
    print("ALL_flood_events_summary.csv 暂不存在。")

if os.path.exists(ts_path):
    display(pd.read_csv(ts_path).head())
else:
    print("ALL_flood_events_timeseries.csv 暂不存在。")

if os.path.exists(log_path):
    display(pd.read_csv(log_path).head(20))
else:
    print("batch_log.csv 暂不存在。")


,source_file,station_name,year,event_id,rising_time,rising_value(ML/day),peak_time,peak_value(ML/day),recession_time,recession_value(ML/day),n_points,duration_days
0,D:\workroom\GPLW\makeUP\downloaded_stations_da...,Barcoo River at Blackall 003303A,1969,Barcoo River at Blackall 003303A_1969_1,1969-10-27 00:00:00.000000000,0.000000,1969-12-26 00:00:00.000000000,354.670163,1969-12-31 00:00:00.000000000,11.145542,66,66
1,D:\workroom\GPLW\makeUP\downloaded_stations_da...,Barcoo River at Blackall 003303A,1970,Barcoo River at Blackall 003303A_1970_2,1970-03-13 00:00:00.000000000,0.172799,1970-03-17 00:00:00.000000000,12256.726920,1970-04-10 00:00:00.000000000,0.172799,29,29
2,D:\workroom\GPLW\makeUP\downloaded_stations_da...,Barcoo River at Blackall 003303A,1971,Barcoo River at Blackall 003303A_1971_3,1971-01-31 00:00:00.000000000,174.181498,1971-02-02 00:00:00.000000000,2884.967458,1971-02-06 00:00:00.000000000,176.859884,7,7
3,D:\workroom\GPLW\makeUP\downloaded_stations_da...,Barcoo River at Blackall 003303A,1972,Barcoo River at Blackall 003303A_1972_4,1972-11-23 00:00:00.000000000,5.356772,1972-11-28 00:00:00.000000000,3750.258977,1972-12-24 00:00:00.000000000,5.615971,32,32
4,D:\workroom\GPLW\makeUP\downloaded_stations_da...,Barcoo River at Blackall 003303A,1973,Barcoo River at Blackall 003303A_1973_5,1973-02-17 00:00:00.000000000,592.268933,1973-02-19 00:00:00.000000000,9665.431541,1973-02-24 00:00:00.000000000,632.358325,8,8


,time,runoff,station_name,event_id,flag
0,1969-10-27 00:00:00.000000000,0.0,Barcoo River at Blackall 003303A,Barcoo River at Blackall 003303A_1969_1,rising
1,1969-10-28 00:00:00.000000000,0.0,Barcoo River at Blackall 003303A,Barcoo River at Blackall 003303A_1969_1,process
2,1969-10-29 00:00:00.000000000,0.0,Barcoo River at Blackall 003303A,Barcoo River at Blackall 003303A_1969_1,process
3,1969-10-30 00:00:00.000000000,0.0,Barcoo River at Blackall 003303A,Barcoo River at Blackall 003303A_1969_1,process
4,1969-10-31 00:00:00.000000000,0.0,Barcoo River at Blackall 003303A,Barcoo River at Blackall 003303A_1969_1,process


,file,status,n_events
0,D:\workroom\GPLW\makeUP\downloaded_stations_da...,ok,56.0
1,D:\workroom\GPLW\makeUP\downloaded_stations_da...,ok,58.0
2,D:\workroom\GPLW\makeUP\downloaded_stations_da...,ok,56.0
3,D:\workroom\GPLW\makeUP\downloaded_stations_da...,ok,57.0
4,D:\workroom\GPLW\makeUP\downloaded_stations_da...,ok,59.0
5,D:\workroom\GPLW\makeUP\downloaded_stations_da...,ok,1.0
6,D:\workroom\GPLW\makeUP\downloaded_stations_da...,ok,1.0
7,D:\workroom\GPLW\makeUP\downloaded_stations_da...,ok,1.0
8,D:\workroom\GPLW\makeUP\downloaded_stations_da...,ok,1.0
9,D:\workroom\GPLW\makeUP\downloaded_stations_da...,no_events,NaN
